In [1]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# 检查显卡
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 详细的GPU信息检查
if torch.cuda.is_available():
    print(f"GPU数量: {torch.cuda.device_count()}")
    print(f"当前GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"cuDNN版本: {torch.backends.cudnn.version()}")
    # 创建一个测试张量来确认GPU可用
    test_tensor = torch.randn(1, 1).to(device)
    print(f"✓ GPU测试成功，张量位置: {test_tensor.device}")
else:
    print("警告: CUDA不可用，将使用CPU训练（速度会很慢）")

Using device: cuda
GPU数量: 1
当前GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA版本: 11.8
cuDNN版本: 90100
✓ GPU测试成功，张量位置: cuda:0


In [2]:
# 路径定义
TRAIN_IMAGE_DIR = r"E:\Dataset\shezhenv3-coco/train/images"
TEST_IMAGE_DIR = r"E:\Dataset\shezhenv3-coco/test/images"
TRAIN_LABEL_PATH = r"E:/Projects/-\data-process/output/train-labels.csv"
TEST_LABEL_PATH = r"E:/Projects/-\data-process/output/test-labels.csv"

# 获取所有图片路径并排序
train_images = sorted(glob.glob(os.path.join(TRAIN_IMAGE_DIR, "*.jpg"))) + \
               sorted(glob.glob(os.path.join(TRAIN_IMAGE_DIR, "*.png")))
test_images = sorted(glob.glob(os.path.join(TEST_IMAGE_DIR, "*.jpg"))) + \
              sorted(glob.glob(os.path.join(TEST_IMAGE_DIR, "*.png")))

print(f"找到训练图片数量: {len(train_images)}")
print(f"找到测试图片数量: {len(test_images)}")

找到训练图片数量: 5404
找到测试图片数量: 553


In [3]:
# 读取标签文件
train_df = pd.read_csv(TRAIN_LABEL_PATH)
test_df = pd.read_csv(TEST_LABEL_PATH)

# 获取所有标签列（排除filename列）
all_label_columns = [col for col in train_df.columns if col != 'filename']

# 计算每个标签的正例比例，并过滤掉正例比例<=0.01的标签
print("\n--- 训练集各标签正例比例统计 ---")
label_pos_weights = {}
label_columns = []  # 只保留正例比例>0.01的标签

for col in all_label_columns:
    pos_count = train_df[col].sum()
    total_count = len(train_df)
    pos_rate = pos_count / total_count if total_count > 0 else 0
    label_pos_weights[col] = pos_rate
    
    if pos_rate > 0.01:
        label_columns.append(col)
        print(f"{col}: {pos_rate:.4f} ({pos_count}/{total_count}) ✓ 保留")
    else:
        print(f"{col}: {pos_rate:.4f} ({pos_count}/{total_count}) ✗ 舍弃")

num_labels = len(label_columns)
print(f"\n保留的标签数量: {num_labels}")
print(f"保留的标签: {label_columns}")

# 创建标签字典：filename -> [label1, label2, ..., labelN] (只包含保留的标签)
train_label_dict = {}
for _, row in train_df.iterrows():
    filename = row['filename']
    labels = [int(row[col]) for col in label_columns]
    train_label_dict[filename] = labels

test_label_dict = {}
for _, row in test_df.iterrows():
    filename = row['filename']
    labels = [int(row[col]) for col in label_columns]
    test_label_dict[filename] = labels

print(f"\n成功读取训练标签: {len(train_label_dict)} 条")
print(f"成功读取测试标签: {len(test_label_dict)} 条")


--- 训练集各标签正例比例统计 ---
jiankangshe: 0.0017 (9/5404) ✗ 舍弃
botaishe: 0.0642 (347/5404) ✓ 保留
hongshe: 0.2165 (1170/5404) ✓ 保留
zishe: 0.0346 (187/5404) ✓ 保留
pangdashe: 0.1116 (603/5404) ✓ 保留
shoushe: 0.0403 (218/5404) ✓ 保留
hongdianshe: 0.1939 (1048/5404) ✓ 保留
liewenshe: 0.2443 (1320/5404) ✓ 保留
chihenshe: 0.1671 (903/5404) ✓ 保留
baitaishe: 0.7650 (4134/5404) ✓ 保留
huangtaishe: 0.1721 (930/5404) ✓ 保留
heitaishe: 0.0191 (103/5404) ✓ 保留
huataishe: 0.0405 (219/5404) ✓ 保留
shenquao: 0.0761 (411/5404) ✓ 保留
shenqutu: 0.0002 (1/5404) ✗ 舍弃
gandanao: 0.0322 (174/5404) ✓ 保留
gandantu: 0.0002 (1/5404) ✗ 舍弃
piweiao: 0.0300 (162/5404) ✓ 保留
piweitu: 0.0000 (0/5404) ✗ 舍弃
xinfeiao: 0.0587 (317/5404) ✓ 保留
xinfeitu: 0.0004 (2/5404) ✗ 舍弃

保留的标签数量: 16
保留的标签: ['botaishe', 'hongshe', 'zishe', 'pangdashe', 'shoushe', 'hongdianshe', 'liewenshe', 'chihenshe', 'baitaishe', 'huangtaishe', 'heitaishe', 'huataishe', 'shenquao', 'gandanao', 'piweiao', 'xinfeiao']

成功读取训练标签: 5404 条
成功读取测试标签: 553 条


In [4]:
class ShenzhenDataset(Dataset):
    def __init__(self, image_paths, label_dict, transform=None):
        """
        Args:
            image_paths: 图片路径列表
            label_dict: 标签字典，key为filename（不含扩展名），value为标签列表
            transform: 图像变换
        """
        self.final_image_paths = []
        self.final_labels = []
        
        skipped_count = 0
        
        # 创建反向映射：从图片文件名到CSV中的filename
        # CSV中格式可能是 "train-A (1000)" 或 "test-10001"
        # 图片文件名可能是 "A (1000).jpg" 或 "10001.jpg" 或 "A (102).jpg"
        filename_mapping = {}
        for csv_filename in label_dict.keys():
            # 处理 "train-A (1000)" -> "A (1000)"
            if csv_filename.startswith('train-'):
                img_filename = csv_filename.replace('train-', '')
                filename_mapping[img_filename] = csv_filename
            elif csv_filename.startswith('test-'):
                # 处理 "test-10001" -> 可能需要匹配 "A (102).jpg" 或其他格式
                # 先尝试直接去掉test-前缀
                img_filename = csv_filename.replace('test-', '')
                filename_mapping[img_filename] = csv_filename
                # 也尝试匹配 "A (xxx)" 格式，提取数字部分
                if img_filename.isdigit():
                    # 如果test-10001，可能对应A (101)这样的格式
                    # 这里先保留原映射，后续在匹配时再处理
                    pass
            else:
                filename_mapping[csv_filename] = csv_filename
        
        for img_path in image_paths:
            # 获取文件名（不含扩展名）
            filename_base = os.path.splitext(os.path.basename(img_path))[0]
            
            # 查找匹配的CSV文件名
            matched_filename = None
            
            # 方法1: 直接匹配
            if filename_base in filename_mapping:
                matched_filename = filename_mapping[filename_base]
            elif filename_base in label_dict:
                matched_filename = filename_base
            else:
                # 方法2: 尝试提取数字部分进行匹配
                # 例如 "A (1000)" -> 提取 "1000"，然后匹配 "test-1000" 或类似格式
                import re
                numbers = re.findall(r'\d+', filename_base)
                if numbers:
                    for num_str in numbers:
                        # 尝试匹配 "test-{num}" 或 "train-A ({num})"
                        test_key = f"test-{num_str}"
                        train_key = f"train-A ({num_str})"
                        if test_key in label_dict:
                            matched_filename = test_key
                            break
                        elif train_key in label_dict:
                            matched_filename = train_key
                            break
            
            if matched_filename and matched_filename in label_dict:
                labels = label_dict[matched_filename]
                # 检查标签是否有效（应该是0或1的列表）
                if len(labels) == num_labels and all(l in [0, 1] for l in labels):
                    self.final_image_paths.append(img_path)
                    self.final_labels.append(labels)
                else:
                    skipped_count += 1
            else:
                skipped_count += 1
        
        self.transform = transform
        print(f"最终有效样本数: {len(self.final_image_paths)}")
        if skipped_count > 0:
            print(f"跳过无效样本数: {skipped_count}")

    def __len__(self):
        return len(self.final_image_paths)

    def __getitem__(self, idx):
        # 读取图像
        img = cv2.imread(self.final_image_paths[idx])
        if img is None:
            raise ValueError(f"无法读取图像: {self.final_image_paths[idx]}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        image = Image.fromarray(img)
        if self.transform:
            image = self.transform(image)
        
        # 标签转为Tensor（二分类，使用float类型）
        label_tensor = torch.tensor(self.final_labels[idx], dtype=torch.float32)
        
        return image, label_tensor

In [5]:
# 创建数据集变换
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 创建数据集
train_dataset = ShenzhenDataset(train_images, train_label_dict, transform=train_transform)
test_dataset = ShenzhenDataset(test_images, test_label_dict, transform=val_transform)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")

最终有效样本数: 5404
最终有效样本数: 553
训练集大小: 5404
测试集大小: 553


In [6]:
import timm

class MultiTaskViTBinary(nn.Module):
    def __init__(self, num_labels, model_name='vit_base_patch16_224'):
        super(MultiTaskViTBinary, self).__init__()
        # 加载预训练的 ViT-Base 模型
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        
        embed_dim = self.backbone.num_features  # Base 默认为 768
        
        # 定义多个独立的二分类任务头（只针对保留的标签）
        self.heads = nn.ModuleList([
            nn.Linear(embed_dim, 1) for _ in range(num_labels)
        ])

    def forward(self, x):
        # 提取全局图像特征
        features = self.backbone(x)
        
        # 并行输出所有保留标签的二分类预测结果
        outputs = [head(features) for head in self.heads]
        return outputs

# 实例化模型
model = MultiTaskViTBinary(num_labels).to(device)
print("模型加载完成，已载入 ImageNet 预训练权重。")

# 确认模型参数在GPU上
if torch.cuda.is_available():
    model_params_on_gpu = next(model.parameters()).is_cuda
    print(f"模型参数是否在GPU上: {model_params_on_gpu}")
    print(f"GPU显存已分配: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"GPU显存已缓存: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

模型加载完成，已载入 ImageNet 预训练权重。
模型参数是否在GPU上: True
GPU显存已分配: 327.35 MB
GPU显存已缓存: 382.00 MB


In [7]:
# 创建DataLoader
batch_size = 16
num_workers = 0 if os.name == 'nt' else 2
pin_memory = False if os.name == 'nt' else True

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                         num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)

print(f"DataLoader设置: num_workers={num_workers}, pin_memory={pin_memory}")
print(f"Batch size: {batch_size}")

# 计算类别权重（用于处理不平衡问题）
# 对于每个保留的标签，计算正负样本的权重
# weight = (负样本数) / (正样本数) 用于正样本
pos_weights_list = []
for col in label_columns:
    pos_count = train_df[col].sum()
    neg_count = len(train_df) - pos_count
    if pos_count > 0:
        pos_weight = neg_count / pos_count
    else:
        pos_weight = 1.0  # 如果没有正样本，权重设为1
    pos_weights_list.append(pos_weight)

# 转换为tensor并移到GPU
pos_weights = torch.tensor(pos_weights_list, dtype=torch.float32).to(device)
print(f"\n类别权重（用于处理不平衡，仅保留的标签）:")
for i, col in enumerate(label_columns):
    print(f"  {col}: {pos_weights_list[i]:.4f}")

# 为每个任务创建单独的损失函数（每个任务的pos_weight是标量）
criteria = []
for i in range(num_labels):
    criterion_i = nn.BCEWithLogitsLoss(pos_weight=pos_weights[i])
    criteria.append(criterion_i)

# 优化器
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

# 启用混合精度训练
scaler = torch.amp.GradScaler('cuda') if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler()
print("\n已启用混合精度训练（AMP）以节省显存")

DataLoader设置: num_workers=0, pin_memory=False
Batch size: 16

类别权重（用于处理不平衡，仅保留的标签）:
  botaishe: 14.5735
  hongshe: 3.6188
  zishe: 27.8984
  pangdashe: 7.9619
  shoushe: 23.7890
  hongdianshe: 4.1565
  liewenshe: 3.0939
  chihenshe: 4.9845
  baitaishe: 0.3072
  huangtaishe: 4.8108
  heitaishe: 51.4660
  huataishe: 23.6758
  shenquao: 12.1484
  gandanao: 30.0575
  piweiao: 32.3580
  xinfeiao: 16.0473

已启用混合精度训练（AMP）以节省显存


In [8]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criteria, device, scaler):
    model.train()
    total_loss = 0
    
    pbar = tqdm(enumerate(loader), total=len(loader), desc="  Training", leave=False)
    
    first_batch_checked = False
    
    for batch_idx, (images, labels) in pbar:
        images, labels = images.to(device), labels.to(device)
        
        if not first_batch_checked:
            print(f"\n      第一个batch诊断:")
            print(f"      images设备: {images.device}, labels设备: {labels.device}")
            print(f"      images形状: {images.shape}, labels形状: {labels.shape}")
            if torch.cuda.is_available():
                print(f"      GPU显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
            first_batch_checked = True
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
            preds = model(images)
            # 计算所有保留任务的损失并累加（每个任务使用对应的损失函数）
            loss = sum([criteria[i](preds[i], labels[:, i:i+1]) for i in range(num_labels)])
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    # 存储每个标签的TP, FP, TN, FN
    tp_list = [0] * num_labels
    fp_list = [0] * num_labels
    tn_list = [0] * num_labels
    fn_list = [0] * num_labels
    total = 0
    
    pbar = tqdm(loader, desc="  Evaluating", leave=False)
    
    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
                preds = model(images)
            
            # 对每个任务计算预测结果（sigmoid后阈值0.5）
            for i in range(num_labels):
                probs = torch.sigmoid(preds[i])
                pred_binary = (probs > 0.5).float()
                true_labels = labels[:, i]
                
                tp_list[i] += ((pred_binary == 1) & (true_labels == 1)).sum().item()
                fp_list[i] += ((pred_binary == 1) & (true_labels == 0)).sum().item()
                tn_list[i] += ((pred_binary == 0) & (true_labels == 0)).sum().item()
                fn_list[i] += ((pred_binary == 0) & (true_labels == 1)).sum().item()
            
            total += labels.size(0)
            pbar.set_postfix(samples=total)
    
    # 计算每个标签的准确率、精确率、召回率、F1
    metrics = {}
    for i, col in enumerate(label_columns):
        tp, fp, tn, fn = tp_list[i], fp_list[i], tn_list[i], fn_list[i]
        accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics[col] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    return metrics

print("训练和评估函数定义完成")

训练和评估函数定义完成


In [20]:
# 开始训练
num_epochs = 20
best_f1 = 0.0

if torch.cuda.is_available():
    print("=" * 60)
    print("训练前GPU状态检查:")
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU显存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"当前显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print("=" * 60)
    print()

for epoch in range(num_epochs):
    # 训练阶段
    train_loss = train_one_epoch(model, train_loader, optimizer, criteria, device, scaler)
    
    # 验证阶段（在测试集上）
    test_metrics = evaluate(model, test_loader, device)
    
    # 计算平均F1分数
    avg_f1 = np.mean([test_metrics[col]['f1'] for col in label_columns])
    avg_acc = np.mean([test_metrics[col]['accuracy'] for col in label_columns])
    
    # 日志打印
    if torch.cuda.is_available() and epoch == 0:
        print(f"  训练后GPU显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    
    print(f"\nEpoch {epoch+1}/{num_epochs} | Loss: {train_loss:.4f}")
    print(f"  平均准确率: {avg_acc:.2%} | 平均F1分数: {avg_f1:.4f}")
    
    # 打印前5个标签的详细指标
    print("  前5个标签指标:")
    for i, col in enumerate(label_columns[:5]):
        m = test_metrics[col]
        print(f"    {col}: Acc={m['accuracy']:.2%}, Prec={m['precision']:.2%}, Rec={m['recall']:.2%}, F1={m['f1']:.4f}")
    
    # 模型保存
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        torch.save(model.state_dict(), "shenzhen_vit_best.pth")
        print(f"  ✓ 保存最佳模型 (F1提升至: {best_f1:.4f})")
    print("-" * 30)

# 最终保存
torch.save(model.state_dict(), "shenzhen_vit_final.pth")
print(f"\n训练完成！最佳平均F1分数: {best_f1:.4f}")

训练前GPU状态检查:
GPU设备: NVIDIA GeForce RTX 4060 Laptop GPU
GPU显存总量: 8.00 GB
当前显存使用: 1529.89 MB



  Training:   0%|          | 0/338 [00:00<?, ?it/s]


      第一个batch诊断:
      images设备: cuda:0, labels设备: cuda:0
      images形状: torch.Size([16, 3, 224, 224]), labels形状: torch.Size([16, 16])
      GPU显存使用: 1539.08 MB


  Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

  训练后GPU显存使用: 2529.14 MB

Epoch 1/20 | Loss: 17.9077
  平均准确率: 63.13% | 平均F1分数: 0.1666
  前5个标签指标:
    botaishe: Acc=82.90%, Prec=0.07%, Rec=6.25%, F1=0.0013
    hongshe: Acc=56.41%, Prec=29.46%, Rec=45.03%, F1=0.3561
    zishe: Acc=46.96%, Prec=3.56%, Rec=59.79%, F1=0.0673
    pangdashe: Acc=45.10%, Prec=6.67%, Rec=61.45%, F1=0.1204
    shoushe: Acc=82.46%, Prec=8.41%, Rec=22.83%, F1=0.1229
  ✓ 保存最佳模型 (F1提升至: 0.1666)
------------------------------


...


Epoch 19/20 | Loss: 1.4995
  平均准确率: 83.19% | 平均F1分数: 0.1675
  前5个标签指标:
    botaishe: Acc=93.83%, Prec=0.19%, Rec=6.25%, F1=0.0037
    hongshe: Acc=67.17%, Prec=31.93%, Rec=19.98%, F1=0.2458
    zishe: Acc=92.58%, Prec=4.65%, Rec=6.76%, F1=0.0551
    pangdashe: Acc=82.62%, Prec=8.19%, Rec=18.06%, F1=0.1127
    shoushe: Acc=89.46%, Prec=9.19%, Rec=10.78%, F1=0.0992
------------------------------


  Training:   0%|          | 0/338 [00:00<?, ?it/s]


      第一个batch诊断:
      images设备: cuda:0, labels设备: cuda:0
      images形状: torch.Size([16, 3, 224, 224]), labels形状: torch.Size([16, 16])
      GPU显存使用: 2538.32 MB


  Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 20/20 | Loss: 1.6682
  平均准确率: 84.81% | 平均F1分数: 0.1670
  前5个标签指标:
    botaishe: Acc=91.83%, Prec=0.14%, Rec=6.25%, F1=0.0028
    hongshe: Acc=70.20%, Prec=33.38%, Rec=11.35%, F1=0.1694
    zishe: Acc=90.69%, Prec=6.48%, Rec=14.23%, F1=0.0891
    pangdashe: Acc=86.33%, Prec=9.31%, Rec=14.15%, F1=0.1123
    shoushe: Acc=92.05%, Prec=9.96%, Rec=5.92%, F1=0.0743
------------------------------

训练完成！最佳平均F1分数: 0.1879


In [21]:
# 计算最优模型在所有标签上的ROC-AUC值
from sklearn.metrics import roc_auc_score

# 加载最佳模型
model.load_state_dict(torch.load("shenzhen_vit_best.pth"))
model.eval()
print("已加载最佳模型，开始计算ROC-AUC值...\n")

# 收集所有预测概率和真实标签
all_pred_probs = [[] for _ in range(num_labels)]
all_true_labels = [[] for _ in range(num_labels)]

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="计算ROC-AUC"):
        images, labels = images.to(device), labels.to(device)
        
        with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
            preds = model(images)
        
        # 对每个任务，计算sigmoid概率并收集
        for i in range(num_labels):
            probs = torch.sigmoid(preds[i]).cpu().numpy().flatten()
            true_vals = labels[:, i].cpu().numpy().flatten()
            all_pred_probs[i].extend(probs)
            all_true_labels[i].extend(true_vals)

# 计算每个标签的ROC-AUC
roc_auc_scores = {}
print("=" * 60)
print("各标签ROC-AUC值:")
print("=" * 60)

for i, col in enumerate(label_columns):
    y_true = np.array(all_true_labels[i])
    y_pred_proba = np.array(all_pred_probs[i])
    
    # 检查是否有正负样本（ROC-AUC需要至少一个正样本和一个负样本）
    unique_labels = np.unique(y_true)
    if len(unique_labels) < 2:
        # 如果只有一个类别，无法计算ROC-AUC
        roc_auc = np.nan
        print(f"{col:20s}: 无法计算 (只有{len(unique_labels)}个类别)")
    else:
        try:
            roc_auc = roc_auc_score(y_true, y_pred_proba)
            roc_auc_scores[col] = roc_auc
            print(f"{col:20s}: {roc_auc:.4f}")
        except ValueError as e:
            roc_auc = np.nan
            print(f"{col:20s}: 计算错误 - {str(e)}")

# 计算平均ROC-AUC（排除NaN值）
valid_scores = [v for v in roc_auc_scores.values() if not np.isnan(v)]
if len(valid_scores) > 0:
    mean_roc_auc = np.mean(valid_scores)
    print("=" * 60)
    print(f"平均ROC-AUC值: {mean_roc_auc:.4f}")
    print(f"(基于{len(valid_scores)}个有效标签)")
else:
    print("=" * 60)
    print("警告: 无法计算平均ROC-AUC值")
print("=" * 60)

已加载最佳模型，开始计算ROC-AUC值...



计算ROC-AUC:   0%|          | 0/35 [00:00<?, ?it/s]

各标签ROC-AUC值:
botaishe            : 0.9982
hongshe             : 0.8382
zishe               : 0.8754
pangdashe           : 0.8595
shoushe             : 0.9360
hongdianshe         : 0.9336
liewenshe           : 0.9004
chihenshe           : 0.7552
baitaishe           : 0.9057
huangtaishe         : 0.9608
heitaishe           : 0.8739
huataishe           : 0.8553
shenquao            : 0.9275
gandanao            : 0.8952
piweiao             : 0.9063
xinfeiao            : 0.9184
平均ROC-AUC值: 0.8962
(基于16个有效标签)


In [22]:
# 计算最优模型在所有标签上的ROC-AUC值
from sklearn.metrics import roc_auc_score

# 加载最佳模型
model.load_state_dict(torch.load("shenzhen_vit_final.pth"))
model.eval()
print("已加载最终模型，开始计算ROC-AUC值...\n")

# 收集所有预测概率和真实标签
all_pred_probs = [[] for _ in range(num_labels)]
all_true_labels = [[] for _ in range(num_labels)]

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="计算ROC-AUC"):
        images, labels = images.to(device), labels.to(device)
        
        with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
            preds = model(images)
        
        # 对每个任务，计算sigmoid概率并收集
        for i in range(num_labels):
            probs = torch.sigmoid(preds[i]).cpu().numpy().flatten()
            true_vals = labels[:, i].cpu().numpy().flatten()
            all_pred_probs[i].extend(probs)
            all_true_labels[i].extend(true_vals)

# 计算每个标签的ROC-AUC
roc_auc_scores = {}
print("=" * 60)
print("各标签ROC-AUC值:")
print("=" * 60)

for i, col in enumerate(label_columns):
    y_true = np.array(all_true_labels[i])
    y_pred_proba = np.array(all_pred_probs[i])
    
    # 检查是否有正负样本（ROC-AUC需要至少一个正样本和一个负样本）
    unique_labels = np.unique(y_true)
    if len(unique_labels) < 2:
        # 如果只有一个类别，无法计算ROC-AUC
        roc_auc = np.nan
        print(f"{col:20s}: 无法计算 (只有{len(unique_labels)}个类别)")
    else:
        try:
            roc_auc = roc_auc_score(y_true, y_pred_proba)
            roc_auc_scores[col] = roc_auc
            print(f"{col:20s}: {roc_auc:.4f}")
        except ValueError as e:
            roc_auc = np.nan
            print(f"{col:20s}: 计算错误 - {str(e)}")

# 计算平均ROC-AUC（排除NaN值）
valid_scores = [v for v in roc_auc_scores.values() if not np.isnan(v)]
if len(valid_scores) > 0:
    mean_roc_auc = np.mean(valid_scores)
    print("=" * 60)
    print(f"平均ROC-AUC值: {mean_roc_auc:.4f}")
    print(f"(基于{len(valid_scores)}个有效标签)")
else:
    print("=" * 60)
    print("警告: 无法计算平均ROC-AUC值")
print("=" * 60)

已加载最终模型，开始计算ROC-AUC值...



计算ROC-AUC:   0%|          | 0/35 [00:00<?, ?it/s]

各标签ROC-AUC值:
botaishe            : 0.9991
hongshe             : 0.8616
zishe               : 0.8726
pangdashe           : 0.8051
shoushe             : 0.9247
hongdianshe         : 0.9327
liewenshe           : 0.9011
chihenshe           : 0.7581
baitaishe           : 0.8753
huangtaishe         : 0.9500
heitaishe           : 0.8033
huataishe           : 0.8128
shenquao            : 0.9470
gandanao            : 0.9158
piweiao             : 0.9286
xinfeiao            : 0.8571
平均ROC-AUC值: 0.8841
(基于16个有效标签)


In [9]:
# ==========================================
# 使用ResNet50进行训练（复用相同的数据集和16个标签）
# ==========================================
from torchvision import models

class MultiTaskResNet50(nn.Module):
    def __init__(self, num_labels):
        super(MultiTaskResNet50, self).__init__()
        # 加载预训练的ResNet50模型
        resnet50 = models.resnet50(pretrained=True)
        # 移除最后的全连接层，保留特征提取器
        self.backbone = nn.Sequential(*list(resnet50.children())[:-1])
        
        # ResNet50的fc层输入维度是2048
        embed_dim = 2048
        
        # 定义多个独立的二分类任务头（只针对保留的标签）
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, 512),
                nn.ReLU(True),
                nn.Dropout(0.5),
                nn.Linear(512, 1)
            ) for _ in range(num_labels)
        ])

    def forward(self, x):
        # 提取特征
        features = self.backbone(x)
        # ResNet50的backbone输出是 [batch_size, 2048, 1, 1]，需要flatten
        features = features.view(features.size(0), -1)
        
        # 并行输出所有保留标签的二分类预测结果
        outputs = [head(features) for head in self.heads]
        return outputs

# 实例化ResNet50模型
resnet_model = MultiTaskResNet50(num_labels).to(device)
print("ResNet50模型加载完成，已载入ImageNet预训练权重。")

# 确认模型参数在GPU上
if torch.cuda.is_available():
    model_params_on_gpu = next(resnet_model.parameters()).is_cuda
    print(f"模型参数是否在GPU上: {model_params_on_gpu}")
    print(f"GPU显存已分配: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"GPU显存已缓存: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

e:\Python\envs\tongue_vit\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Python\envs\tongue_vit\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\13529/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:29<00:00, 3.42MB/s]


ResNet50模型加载完成，已载入ImageNet预训练权重。
模型参数是否在GPU上: True
GPU显存已分配: 481.58 MB
GPU显存已缓存: 540.00 MB


In [10]:
# ResNet50的训练设置（复用相同的损失函数和优化器设置）
# 类别权重已经在之前计算好了，直接复用
print(f"ResNet50训练设置:")
print(f"  Batch size: {batch_size}")
print(f"  学习率: 5e-5")
print(f"  已启用混合精度训练")

# 为ResNet50创建优化器
resnet_optimizer = torch.optim.AdamW(resnet_model.parameters(), lr=5e-5, weight_decay=0.01)

# 复用相同的损失函数列表（criteria已经在之前定义）
print(f"\n使用相同的{num_labels}个标签进行训练")

ResNet50训练设置:
  Batch size: 16
  学习率: 5e-5
  已启用混合精度训练

使用相同的16个标签进行训练


In [11]:
# ResNet50训练函数（复用相同的训练和评估函数结构）
def train_resnet_epoch(model, loader, optimizer, criteria, device, scaler):
    model.train()
    total_loss = 0
    
    pbar = tqdm(enumerate(loader), total=len(loader), desc="  Training ResNet50", leave=False)
    
    first_batch_checked = False
    
    for batch_idx, (images, labels) in pbar:
        images, labels = images.to(device), labels.to(device)
        
        if not first_batch_checked:
            print(f"\n      第一个batch诊断:")
            print(f"      images设备: {images.device}, labels设备: {labels.device}")
            print(f"      images形状: {images.shape}, labels形状: {labels.shape}")
            if torch.cuda.is_available():
                print(f"      GPU显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
            first_batch_checked = True
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
            preds = model(images)
            # 计算所有保留任务的损失并累加（每个任务使用对应的损失函数）
            loss = sum([criteria[i](preds[i], labels[:, i:i+1]) for i in range(num_labels)])
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return total_loss / len(loader)

def evaluate_resnet(model, loader, device):
    model.eval()
    # 存储每个标签的TP, FP, TN, FN
    tp_list = [0] * num_labels
    fp_list = [0] * num_labels
    tn_list = [0] * num_labels
    fn_list = [0] * num_labels
    total = 0
    
    pbar = tqdm(loader, desc="  Evaluating ResNet50", leave=False)
    
    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
                preds = model(images)
            
            # 对每个任务计算预测结果（sigmoid后阈值0.5）
            for i in range(num_labels):
                probs = torch.sigmoid(preds[i])
                pred_binary = (probs > 0.5).float()
                true_labels = labels[:, i]
                
                tp_list[i] += ((pred_binary == 1) & (true_labels == 1)).sum().item()
                fp_list[i] += ((pred_binary == 1) & (true_labels == 0)).sum().item()
                tn_list[i] += ((pred_binary == 0) & (true_labels == 0)).sum().item()
                fn_list[i] += ((pred_binary == 0) & (true_labels == 1)).sum().item()
            
            total += labels.size(0)
            pbar.set_postfix(samples=total)
    
    # 计算每个标签的准确率、精确率、召回率、F1
    metrics = {}
    for i, col in enumerate(label_columns):
        tp, fp, tn, fn = tp_list[i], fp_list[i], tn_list[i], fn_list[i]
        accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics[col] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }
    
    return metrics

print("ResNet50训练和评估函数定义完成")

ResNet50训练和评估函数定义完成


In [12]:
# 开始训练ResNet50
resnet_num_epochs = 20
resnet_best_f1 = 0.0

if torch.cuda.is_available():
    print("=" * 60)
    print("ResNet50训练前GPU状态检查:")
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")
    print(f"GPU显存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"当前显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print("=" * 60)
    print()

for epoch in range(resnet_num_epochs):
    # 训练阶段
    train_loss = train_resnet_epoch(resnet_model, train_loader, resnet_optimizer, 
                                    criteria, device, scaler)
    
    # 验证阶段（在测试集上）
    test_metrics = evaluate_resnet(resnet_model, test_loader, device)
    
    # 计算平均F1分数
    avg_f1 = np.mean([test_metrics[col]['f1'] for col in label_columns])
    avg_acc = np.mean([test_metrics[col]['accuracy'] for col in label_columns])
    
    # 日志打印
    if torch.cuda.is_available() and epoch == 0:
        print(f"  训练后GPU显存使用: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    
    print(f"\nEpoch {epoch+1}/{resnet_num_epochs} | Loss: {train_loss:.4f}")
    print(f"  平均准确率: {avg_acc:.2%} | 平均F1分数: {avg_f1:.4f}")
    
    # 打印前5个标签的详细指标
    print("  前5个标签指标:")
    for i, col in enumerate(label_columns[:5]):
        m = test_metrics[col]
        print(f"    {col}: Acc={m['accuracy']:.2%}, Prec={m['precision']:.2%}, Rec={m['recall']:.2%}, F1={m['f1']:.4f}")
    
    # 模型保存
    if avg_f1 > resnet_best_f1:
        resnet_best_f1 = avg_f1
        torch.save(resnet_model.state_dict(), "shenzhen_resnet50_best.pth")
        print(f"  ✓ 保存最佳模型 (F1提升至: {resnet_best_f1:.4f})")
    print("-" * 30)

# 最终保存
torch.save(resnet_model.state_dict(), "shenzhen_resnet50_final.pth")
print(f"\nResNet50训练完成！最佳平均F1分数: {resnet_best_f1:.4f}")

ResNet50训练前GPU状态检查:
GPU设备: NVIDIA GeForce RTX 4060 Laptop GPU
GPU显存总量: 8.00 GB
当前显存使用: 481.58 MB



  Training ResNet50:   0%|          | 0/338 [00:00<?, ?it/s]


      第一个batch诊断:
      images设备: cuda:0, labels设备: cuda:0
      images形状: torch.Size([16, 3, 224, 224]), labels形状: torch.Size([16, 16])
      GPU显存使用: 490.77 MB


  Evaluating ResNet50:   0%|          | 0/35 [00:00<?, ?it/s]

  训练后GPU显存使用: 968.01 MB

Epoch 1/20 | Loss: 16.7962
  平均准确率: 70.69% | 平均F1分数: 0.1792
  前5个标签指标:
    botaishe: Acc=80.42%, Prec=0.23%, Rec=25.00%, F1=0.0046
    hongshe: Acc=59.62%, Prec=27.71%, Rec=31.59%, F1=0.2953
    zishe: Acc=73.84%, Prec=5.47%, Rec=44.13%, F1=0.0974
    pangdashe: Acc=59.01%, Prec=7.09%, Rec=47.11%, F1=0.1232
    shoushe: Acc=83.48%, Prec=8.28%, Rec=20.51%, F1=0.1179
  ✓ 保存最佳模型 (F1提升至: 0.1792)
------------------------------


...


Epoch 19/20 | Loss: 3.3039
  平均准确率: 81.32% | 平均F1分数: 0.1750
  前5个标签指标:
    botaishe: Acc=94.38%, Prec=0.21%, Rec=6.25%, F1=0.0040
    hongshe: Acc=67.55%, Prec=30.15%, Rec=16.11%, F1=0.2100
    zishe: Acc=89.11%, Prec=4.20%, Rec=11.03%, F1=0.0608
    pangdashe: Acc=67.79%, Prec=7.24%, Rec=36.13%, F1=0.1206
    shoushe: Acc=88.24%, Prec=8.46%, Rec=12.05%, F1=0.0994
------------------------------


  Training ResNet50:   0%|          | 0/338 [00:00<?, ?it/s]


      第一个batch诊断:
      images设备: cuda:0, labels设备: cuda:0
      images形状: torch.Size([16, 3, 224, 224]), labels形状: torch.Size([16, 16])
      GPU显存使用: 977.19 MB


  Evaluating ResNet50:   0%|          | 0/35 [00:00<?, ?it/s]


Epoch 20/20 | Loss: 2.9400
  平均准确率: 82.84% | 平均F1分数: 0.1683
  前5个标签指标:
    botaishe: Acc=95.88%, Prec=0.85%, Rec=18.75%, F1=0.0163
    hongshe: Acc=66.76%, Prec=28.09%, Rec=15.48%, F1=0.1996
    zishe: Acc=89.15%, Prec=3.33%, Rec=8.54%, F1=0.0480
    pangdashe: Acc=71.31%, Prec=6.84%, Rec=29.24%, F1=0.1108
    shoushe: Acc=89.43%, Prec=8.24%, Rec=9.51%, F1=0.0883
------------------------------

ResNet50训练完成！最佳平均F1分数: 0.1859


In [14]:
# 计算ResNet50最优模型在所有标签上的ROC-AUC值
from sklearn.metrics import roc_auc_score

print("\n" + "=" * 60)
print("计算ResNet50最优模型的ROC-AUC值")
print("=" * 60)

# 加载最佳模型
resnet_model.load_state_dict(torch.load("shenzhen_resnet50_best.pth"))
resnet_model.eval()
print("已加载ResNet50最佳模型，开始计算ROC-AUC值...\n")

# 收集所有预测概率和真实标签
all_pred_probs_resnet = [[] for _ in range(num_labels)]
all_true_labels_resnet = [[] for _ in range(num_labels)]

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="计算ROC-AUC"):
        images, labels = images.to(device), labels.to(device)
        
        with torch.amp.autocast('cuda') if hasattr(torch.amp, 'autocast') else torch.cuda.amp.autocast():
            preds = resnet_model(images)
        
        # 对每个任务，计算sigmoid概率并收集
        for i in range(num_labels):
            probs = torch.sigmoid(preds[i]).cpu().numpy().flatten()
            true_vals = labels[:, i].cpu().numpy().flatten()
            all_pred_probs_resnet[i].extend(probs)
            all_true_labels_resnet[i].extend(true_vals)

# 计算每个标签的ROC-AUC
roc_auc_scores_resnet = {}
print("=" * 60)
print("ResNet50各标签ROC-AUC值:")
print("=" * 60)

for i, col in enumerate(label_columns):
    y_true = np.array(all_true_labels_resnet[i])
    y_pred_proba = np.array(all_pred_probs_resnet[i])
    
    # 检查是否有正负样本（ROC-AUC需要至少一个正样本和一个负样本）
    unique_labels = np.unique(y_true)
    if len(unique_labels) < 2:
        # 如果只有一个类别，无法计算ROC-AUC
        roc_auc = np.nan
        print(f"{col:20s}: 无法计算 (只有{len(unique_labels)}个类别)")
    else:
        try:
            roc_auc = roc_auc_score(y_true, y_pred_proba)
            roc_auc_scores_resnet[col] = roc_auc
            print(f"{col:20s}: {roc_auc:.4f}")
        except ValueError as e:
            roc_auc = np.nan
            print(f"{col:20s}: 计算错误 - {str(e)}")

# 计算平均ROC-AUC（排除NaN值）
valid_scores_resnet = [v for v in roc_auc_scores_resnet.values() if not np.isnan(v)]
if len(valid_scores_resnet) > 0:
    mean_roc_auc_resnet = np.mean(valid_scores_resnet)
    print("=" * 60)
    print(f"ResNet50平均ROC-AUC值: {mean_roc_auc_resnet:.4f}")
    print(f"(基于{len(valid_scores_resnet)}个有效标签)")
else:
    print("=" * 60)
    print("警告: 无法计算平均ROC-AUC值")
print("=" * 60)


计算ResNet50最优模型的ROC-AUC值
已加载ResNet50最佳模型，开始计算ROC-AUC值...



计算ROC-AUC:   0%|          | 0/35 [00:00<?, ?it/s]

ResNet50各标签ROC-AUC值:
botaishe            : 1.0000
hongshe             : 0.8965
zishe               : 0.8928
pangdashe           : 0.7875
shoushe             : 0.9497
hongdianshe         : 0.9289
liewenshe           : 0.8976
chihenshe           : 0.8082
baitaishe           : 0.8814
huangtaishe         : 0.9582
heitaishe           : 0.6552
huataishe           : 0.9128
shenquao            : 0.9588
gandanao            : 0.8127
piweiao             : 0.9075
xinfeiao            : 0.7917
ResNet50平均ROC-AUC值: 0.8775
(基于16个有效标签)
